In [8]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [4]:
df = pd.read_csv('homework_7.1.csv')
df.drop(columns=['Unnamed: 0'], inplace=True)
df

,X,W,Z,Y
0,1.137055,1.221768,0.327829,1.944532
1,-0.112905,0.465835,0.599650,0.655514
2,2.077755,1.795414,-0.063393,5.934411
3,0.456373,-0.512159,1.177413,-0.188064
4,-1.012402,0.080002,-0.275697,-0.533775
...,...,...,...,...
9995,2.569934,1.233620,0.930467,6.188783
9996,0.190150,1.022164,-0.015151,0.697187
9997,-1.184465,-1.475929,-0.287056,-1.575303
9998,-0.121286,-0.914357,1.706237,-1.809819


Q1. Suppose that a process can be modeled via linear regression: 

```
W = np.random.normal(0, 1, (1000,))
X = W + np.random.normal(0, 1, (1000,)) 
Z = np.random.normal(0, 1, (1000,)) 
Y = X + Z + W + np.random.normal(0, 1, (1000,))
```

Which is closest to the correlation of *X* with the error term in the equation for *Y*? 

Option A: -0.50

Option B: 0

Option C: 1.0

Option D: 0.75

Option E: 0.5


In [9]:
W = np.random.normal(0, 1, (1000,))
X = W + np.random.normal(0, 1, (1000,)) 
Z = np.random.normal(0, 1, (1000,)) 
Y = X + Z + W + np.random.normal(0, 1, (1000,))

In [17]:
structural_error = Y - X
np.corrcoef(X, structural_error)

array([[1.        , 0.40700471],
       [0.40700471, 1.        ]])

A1. Option E (0.5)

Q2: If *Y* is written as depending on *X* and *Z* only, *W* is part of the error term. Which, then, is closest to the expected correlation of *X* with the error term in the equation for *Y*?

Option A: 0.75

Option B: 0.50

Option C: 1.0

Option D: -0.50

Option E: 0

In [22]:
structural_error = Y - X - Z

# Calculate the correlation between X and the structural error
corr_matrix = np.corrcoef(X, structural_error)
corr_matrix

array([[1.        , 0.49556515],
       [0.49556515, 1.        ]])

A2. Option B. (0.5)

Q3. In the data frame for homework_7.1.csv, control for W by regressing *Y* on X
 and ﻿Z﻿ at the following constant values of ﻿W﻿: -1, 0, and 1. (You cannot literally use a constant value of ﻿W﻿, of course, or you will have only one data point! How will you manage this?) The question is: Is the coefficient of ﻿X﻿

Option A. decreasing

Option B. staying about the same (say, within 0.2 or so) as﻿W﻿ increases? 

Option C. increasing

In [23]:
# 2. Define a narrow window to simulate holding W constant
tolerance = 0.05
w_targets = [-1, 0, 1]

print("--- Regression Results within W Slices ---")

for w_val in w_targets:
    # Create a boolean mask for the current slice (e.g., -1.05 <= W <= -0.95)
    mask = (W >= w_val - tolerance) & (W <= w_val + tolerance)
    
    # Isolate variables inside the slice
    X_slice = X[mask]
    Z_slice = Z[mask]
    Y_slice = Y[mask]
    
    # Set up and fit the OLS regression: Y = b0 + b1*X + b2*Z
    features = np.column_stack((X_slice, Z_slice))
    features = sm.add_constant(features)
    model = sm.OLS(Y_slice, features).fit()
    
    # params[0] is intercept, params[1] is X coefficient, params[2] is Z coefficient
    x_coef = model.params[1] 
    sample_count = np.sum(mask)
    
    print(f"Slice W ≈ {w_val:2d} (n={sample_count:,}) | X Coefficient: {x_coef:.4f}")

--- Regression Results within W Slices ---
Slice W ≈ -1 (n=23) | X Coefficient: 0.9195
Slice W ≈  0 (n=31) | X Coefficient: 1.2042
Slice W ≈  1 (n=28) | X Coefficient: 1.1052


A3. Option B.

Q4. 

```
def make_error(corr_const, num): 
          err = list() 
          prev = np.random.normal(0, 1) 
          for n in range(num): 
                   prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 
                   err.append(prev) 
          return np.array(err) 
```

Create a linear regression model that uses this function as the error for both (a) the treatment, ﻿X﻿, and (b) the outcome, ﻿Y﻿. (You can use random normal error for any other covariates, if you have them.) 

As corr_const increases from 0.2 to 0.5 to 0.8, find (i) the standard deviation of the estimate of the ﻿X﻿ coefficient over many trials, and (ii) the mean of the standard error estimate of the ﻿X﻿ coefficient over many trials. 

When corr_const increases, then: 

Hint: don't forget to include an intercept in your regression

Option A: (i) and (ii) differ, but their ratio remains about the same

Option B: The ratio (i) / (ii) decreases

Option C: The ratio (i) / (ii) increases

Option D: (i) and (ii) remain about the same

In [24]:
def make_error(corr_const, num): 
          err = list() 
          prev = np.random.normal(0, 1) 
          for n in range(num): 
                   prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 
                   err.append(prev) 
          return np.array(err) 

In [25]:
# Run 1,000 trials to calculate (i) actual SD and (ii) mean OLS Standard Error
for c in [0.2, 0.5, 0.8]:
    estimates, std_errors = [], []
    for _ in range(1000):
        X = make_error(c, 1000)
        Y = X + make_error(c, 1000)  # DGP: Y = 1*X + error
        
        model = sm.OLS(Y, sm.add_constant(X)).fit()
        estimates.append(model.params[1])
        std_errors.append(model.bse[1])
        
    actual_sd = np.std(estimates)
    mean_se = np.mean(std_errors)
    print(f"corr_const = {c} | (i) Actual SD: {actual_sd:.4f} | (ii) Mean SE: {mean_se:.4f} | Ratio: {actual_sd/mean_se:.2f}")

corr_const = 0.2 | (i) Actual SD: 0.0323 | (ii) Mean SE: 0.0316 | Ratio: 1.02
corr_const = 0.5 | (i) Actual SD: 0.0411 | (ii) Mean SE: 0.0317 | Ratio: 1.30
corr_const = 0.8 | (i) Actual SD: 0.0695 | (ii) Mean SE: 0.0316 | Ratio: 2.20


A4. Option C